# Layer 4 — Prescriptive Analytics: Real-Time Escalation Risk Score

**Portfolio:** Mario Casanova — Analytics Engineering for Enterprise Support Operations  
**Objective:** Build a classifier that scores the escalation probability of any open ticket, enabling Resolution Managers to intervene *before* a case escalates and damages NPS.

---

## Business Problem

Escalations are expensive: they consume executive attention, damage customer NPS, and can trigger SLA penalty clauses. The GSO currently identifies at-risk cases reactively — only after a customer requests escalation.

An **Escalation Risk Score** transforms this into a proactive workflow:
1. Every open ticket is scored every 4 hours
2. Tickets scoring ≥0.7 probability trigger an automatic Resolution Manager review
3. The RM can proactively contact the customer and reallocate resources

## Model Design

| Feature Category | Features Used |
|---|---|
| Time | Hours since ticket open, SLA breach flag, SLA hours remaining |
| Priority & Tier | Priority (P1-P4), customer tier (Enterprise, Strategic) |
| Product | platform software version age (days since release), migration status |
| History | Engineer's historical escalation rate, cluster's prior incidents |

**Models compared:** Logistic Regression (interpretable) vs. Random Forest (higher accuracy) — ensemble approach for production.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, f1_score
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
print('Libraries loaded.')

In [ ]:
tickets_path = '../data/synthetic/support_tickets.csv'

if os.path.exists(tickets_path):
    df = pd.read_csv(tickets_path, parse_dates=['created_at', 'resolved_at'])
else:
    from src.data_generator import generate_support_tickets
    df = generate_support_tickets()
    df.to_csv(tickets_path, index=False)

print(f'Dataset: {df.shape}')
print(f'Escalation rate: {df.escalated.mean():.2%}')

## 1. Feature Engineering

In [ ]:
SLA_HOURS = {'P1': 0.5, 'P2': 2.0, 'P3': 4.0, 'P4': 8.0}

# AOS version release dates (approximate, for version age calculation)
AOS_RELEASE_DATES = {
    '6.1.1': '2021-06-01', '6.1.2': '2021-09-01', '6.5': '2022-03-01',
    '6.5.1': '2022-06-01', '6.5.2': '2022-09-01', '6.7': '2023-01-01',
    '6.7.1': '2023-04-01', '6.8': '2023-09-01', '6.8.1': '2024-01-01',
    '6.8.2': '2024-06-01'
}

features = df.copy()

# Priority encoding
priority_map = {'P1': 1, 'P2': 2, 'P3': 3, 'P4': 4}
features['priority_num'] = features['priority'].map(priority_map)

# SLA threshold for this ticket's priority
features['sla_threshold_hours'] = features['priority'].map(SLA_HOURS)

# How much of the SLA was consumed at resolution (>1 = breached)
features['sla_consumption_ratio'] = features['ttr_hours'] / features['sla_threshold_hours']

# AOS version age in days at time of ticket creation
features['aos_release_date'] = pd.to_datetime(
    features['aos_version'].map(AOS_RELEASE_DATES)
)
features['aos_version_age_days'] = (
    features['created_at'] - features['aos_release_date']
).dt.days.clip(lower=0)

# Customer tier encoding
tier_map = {'SMB': 0, 'Mid-Market': 1, 'Enterprise': 2, 'Strategic': 3}
features['customer_tier_num'] = features['customer_tier'].map(tier_map)

# Migration multiplier
features['is_migrating_int'] = features['is_migrating'].astype(int)
features['is_vmware_migration'] = (features['migration_type'] == 'VMware-to-AHV').astype(int)

# Engineer's historical escalation rate (global rate per engineer)
eng_escalation_rate = df.groupby('assigned_engineer')['escalated'].mean().rename('eng_esc_rate')
features = features.join(eng_escalation_rate, on='assigned_engineer')

# Cluster's historical incident count (proxy for cluster instability)
cluster_incident_count = df.groupby('cluster_id').size().rename('cluster_incident_count')
features = features.join(cluster_incident_count, on='cluster_id')

# Time-based features
features['created_hour']    = features['created_at'].dt.hour
features['created_weekday'] = features['created_at'].dt.weekday
features['is_weekend']      = (features['created_weekday'] >= 5).astype(int)
features['is_outside_hours'] = (
    (features['created_hour'] < 6) | (features['created_hour'] > 22)
).astype(int)

# Region risk index (based on SLA compliance from Layer 1 analysis)
# Higher = more at-risk region
region_risk = {
    'AMER-EAST': 0.2, 'AMER-WEST': 0.2, 'LATAM': 0.6, 'EMEA-WEST': 0.3,
    'EMEA-EAST': 0.4, 'APAC-ANZ': 0.3, 'APAC-INDIA': 0.5
}
features['region_risk'] = features['region'].map(region_risk)

FEATURE_COLS = [
    'priority_num', 'sla_threshold_hours', 'sla_consumption_ratio',
    'ttr_hours', 'aos_version_age_days', 'customer_tier_num',
    'is_migrating_int', 'is_vmware_migration',
    'eng_esc_rate', 'cluster_incident_count',
    'created_hour', 'is_weekend', 'is_outside_hours',
    'region_risk', 'sla_breached'
]

TARGET = 'escalated'

X = features[FEATURE_COLS].fillna(0)
y = features[TARGET].astype(int)

print(f'Feature matrix: {X.shape}')
print(f'Positive class (escalated): {y.sum():,} ({y.mean():.2%})')

## 2. Class Imbalance Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
class_counts = y.value_counts()
axes[0].bar(['Not Escalated', 'Escalated'], class_counts.values,
            color=['#4C72B0', '#C44E52'], alpha=0.85)
axes[0].set_title('Class Distribution (Target Variable)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(y):.1%})', ha='center', fontsize=10)

# Feature correlation with target
corr_with_target = X.corrwith(y.astype(float)).sort_values()
corr_with_target.plot(kind='barh', ax=axes[1], color=[
    '#C44E52' if v > 0 else '#4C72B0' for v in corr_with_target
], alpha=0.85)
axes[1].set_title('Feature Correlation with Escalation Target', fontweight='bold')
axes[1].set_xlabel('Pearson r')
axes[1].axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

imbalance_ratio = class_counts[0] / class_counts[1]
print(f'Class imbalance ratio: {imbalance_ratio:.1f}:1 (non-escalated:escalated)')
print('Strategy: class_weight="balanced" in models to handle imbalance.')

## 3. Model Training — Logistic Regression vs. Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Train escalation rate: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

# ── Logistic Regression ──
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced', max_iter=1000, C=0.1, random_state=RANDOM_STATE
    ))
])
lr_pipeline.fit(X_train, y_train)

# ── Random Forest ──
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train, y_train)

print('\nModels trained.')

## 4. Model Evaluation — ROC AUC and Precision-Recall

In [ ]:
# Predictions
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]
rf_proba = rf_model.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(y_test, lr_proba)
rf_auc = roc_auc_score(y_test, rf_proba)
lr_ap  = average_precision_score(y_test, lr_proba)
rf_ap  = average_precision_score(y_test, rf_proba)

print(f'Logistic Regression — AUC-ROC: {lr_auc:.4f}  |  AP: {lr_ap:.4f}')
print(f'Random Forest       — AUC-ROC: {rf_auc:.4f}  |  AP: {rf_ap:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
for name, proba, auc_score, color in [
    ('Logistic Regression', lr_proba, lr_auc, '#4C72B0'),
    ('Random Forest',       rf_proba, rf_auc, '#DD8452')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc_score:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Escalation Risk Classifier', fontweight='bold')
axes[0].legend()

# Precision-Recall Curve
for name, proba, ap_score, color in [
    ('Logistic Regression', lr_proba, lr_ap, '#4C72B0'),
    ('Random Forest',       rf_proba, rf_ap, '#DD8452')
]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP={ap_score:.3f})')
axes[1].axhline(y_test.mean(), color='k', linestyle='--', linewidth=1,
                label=f'Baseline rate ({y_test.mean():.1%})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.suptitle('Model Comparison — Escalation Risk Score', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Threshold Optimization

For escalation prevention, **Recall** matters more than Precision: we'd rather flag 10 false positives (unnecessary RM reviews) than miss one true escalation that damages a strategic account's NPS. We optimize for F1 but evaluate at multiple thresholds.

In [ ]:
# Use Random Forest as the primary model (higher AUC)
thresholds = np.arange(0.2, 0.9, 0.05)
threshold_results = []

for thresh in thresholds:
    preds = (rf_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel() if preds.sum() > 0 else (len(y_test), 0, 0, 0)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    threshold_results.append({
        'threshold': thresh, 'precision': precision, 'recall': recall, 'f1': f1,
        'flagged_pct': preds.mean() * 100
    })

thresh_df = pd.DataFrame(threshold_results)
optimal_thresh = thresh_df.loc[thresh_df['f1'].idxmax()]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='#4C72B0', linewidth=2)
ax.plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall',    color='#C44E52', linewidth=2)
ax.plot(thresh_df['threshold'], thresh_df['f1'],        label='F1 Score',  color='#55A868', linewidth=2)
ax.axvline(optimal_thresh['threshold'], color='black', linestyle='--',
           label=f'Optimal threshold: {optimal_thresh["threshold"]:.2f} (F1={optimal_thresh["f1"]:.3f})')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision-Recall-F1 vs. Classification Threshold (Random Forest)', fontweight='bold')
ax.legend()
ax.set_xlim(0.2, 0.85)
plt.tight_layout()
plt.show()

print(f'\nOptimal threshold: {optimal_thresh["threshold"]:.2f}')
print(f'  Precision:  {optimal_thresh["precision"]:.3f}')
print(f'  Recall:     {optimal_thresh["recall"]:.3f}')
print(f'  F1 Score:   {optimal_thresh["f1"]:.3f}')
print(f'  % Flagged:  {optimal_thresh["flagged_pct"]:.1f}% of tickets would trigger RM review')

## 6. Feature Importance (Interpretability)

In [ ]:
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

# Logistic Regression coefficients (standardized)
lr_coef = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': lr_pipeline.named_steps['model'].coef_[0]
}).sort_values('coefficient', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest importance
axes[0].barh(importance_df['feature'], importance_df['importance'],
             color='#DD8452', alpha=0.85)
axes[0].set_title('Random Forest — Feature Importance\n(Gini impurity reduction)', fontweight='bold')
axes[0].set_xlabel('Importance')

# Logistic Regression coefficients
colors = ['#C44E52' if c > 0 else '#4C72B0' for c in lr_coef['coefficient']]
axes[1].barh(lr_coef['feature'], lr_coef['coefficient'], color=colors, alpha=0.85)
axes[1].set_title('Logistic Regression — Feature Coefficients\n(red = increases escalation risk)', fontweight='bold')
axes[1].set_xlabel('Coefficient (standardized)')
axes[1].axvline(0, color='black', linewidth=0.5)

plt.suptitle('Feature Interpretability: Why Does a Ticket Get Flagged?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Operational Output: Score Card Sample

In [ ]:
# Simulate live scoring output: show top 10 highest-risk open tickets
OPTIMAL_THRESHOLD = optimal_thresh['threshold']

# Add scores back to dataset
all_scores = rf_model.predict_proba(X)[:, 1]
features['escalation_risk_score'] = all_scores
features['risk_flag'] = (all_scores >= OPTIMAL_THRESHOLD).astype(int)

# Simulate 'open' tickets = those in last 30 days
open_mask = features['created_at'] >= (features['created_at'].max() - pd.Timedelta(days=30))
open_tickets = features[open_mask].copy()

risk_cols = [
    'ticket_id', 'priority', 'customer_tier', 'region',
    'migration_type', 'ttr_hours', 'sla_breached',
    'escalation_risk_score', 'risk_flag'
]

top_risk = open_tickets.sort_values('escalation_risk_score', ascending=False).head(10)[risk_cols]
top_risk['escalation_risk_score'] = top_risk['escalation_risk_score'].round(3)
top_risk['risk_label'] = top_risk['escalation_risk_score'].apply(
    lambda x: '🔴 HIGH' if x >= 0.7 else ('🟡 MEDIUM' if x >= 0.4 else '🟢 LOW')
)

print('TOP 10 HIGHEST-RISK OPEN TICKETS — Recommended for RM Review:')
print(top_risk.to_string(index=False))

## 8. Final Model Summary

In [ ]:
top_3_features = importance_df.tail(3)['feature'].tolist()[::-1]

print(f"""
╔══════════════════════════════════════════════════════════════════════════╗
║          PRESCRIPTIVE ANALYTICS — ESCALATION RISK SCORE SUMMARY         ║
╠══════════════════════════════════════════════════════════════════════════╣
║  Primary Model:  Random Forest (200 trees, max_depth=8)                 ║
║  Fallback Model: Logistic Regression (interpretable for audit)           ║
║                                                                          ║
║  PERFORMANCE METRICS                                                     ║
║  ├─ AUC-ROC (RF):   {rf_auc:.4f}                                       ║
║  ├─ AUC-ROC (LR):   {lr_auc:.4f}                                       ║
║  ├─ Optimal Threshold: {optimal_thresh['threshold']:.2f}                            ║
║  ├─ Precision @ threshold: {optimal_thresh['precision']:.3f}                          ║
║  ├─ Recall @ threshold:    {optimal_thresh['recall']:.3f}                          ║
║  └─ F1 @ threshold:        {optimal_thresh['f1']:.3f}                          ║
║                                                                          ║
║  TOP 3 RISK DRIVERS (Random Forest)                                      ║
║  1. {top_3_features[0]:<66}║
║  2. {top_3_features[1]:<66}║
║  3. {top_3_features[2]:<66}║
║                                                                          ║
║  OPERATIONAL WORKFLOW                                                    ║
║  ├─ Score all open tickets every 4 hours via automated pipeline          ║
║  ├─ Tickets ≥{optimal_thresh['threshold']:.2f} probability → RM review queue in Salesforce       ║
║  ├─ {optimal_thresh['flagged_pct']:.1f}% of active tickets flagged (manageable load for RMs)      ║
║  └─ Expected NPS improvement: 0.5–1.2 points by reducing surprise esc.  ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---

## Portfolio Complete

| Layer | Focus | Technique | Business Output |
|---|---|---|---|
| **1. Descriptive** | What is happening | TTR, SLA, NPS, Backlog | GSO Health Dashboard |
| **2. Diagnostic** | Why it's happening | STL + GESD Anomaly Detection | Pulse Alert System |
| **3. Predictive** | What will happen | SARIMA Forecasting + Ljung-Box | 18-Month Staffing Plan |
| **4. Prescriptive** | What to do about it | Random Forest Risk Classifier | Escalation Prevention |

---

**← Previous:** [Layer 3: Predictive — Ticket Volume Forecasting](./03_layer3_predictive_ticket_forecasting.ipynb)

---
*Mario Casanova | Analytics Engineering Portfolio | Enterprise Cloud Infrastructure Support Organization*